In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Візуалізація часового розкладу схеми

<Accordion>
<AccordionItem title="Версії пакетів">

Код на цій сторінці було розроблено з використанням таких вимог.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Хоча [відображувач часової шкали](/guides/visualize-circuit-timing), вбудований у Qiskit, корисний для статичних схем, він може не точно відображати часовий розклад [динамічних схем](/guides/classical-feedforward-and-control-flow) через неявні операції, такі як мовлення та визначення гілок. У рамках підтримки динамічних схем Qiskit Runtime повертає точну інформацію про часовий розклад схеми всередині результатів завдання при запиті.

> **Note:** - Це є експериментальною функцією. Вона має статус попереднього випуску і тому може змінюватися.
> - Ця функція застосовується лише до завдань Qiskit Runtime Sampler.
> - Хоча загальний час схеми повертається в метаданих "compilation", це НЕ час, що використовується для виставлення рахунків (час QPU).
### Увімкнення отримання даних часового розкладу
Щоб увімкнути отримання даних часового розкладу, встанови експериментальний прапор `scheduler_timing` на `True` при запуску примітивного завдання.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Доступ до даних часового розкладу схеми
При запиті дані часового розкладу схеми для кожного PUB повертаються в метаданих результату завдання в полі `["compilation"]["scheduler_timing"]["timing"]`. Це поле містить необроблену інформацію про часовий розклад. Для відображення інформації про часовий розклад використовуй вбудований інструмент візуалізації, як описано в розділі [Візуалізація часових розкладів](#visualize-timings).

Використовуй наступний код для доступу до даних часового розкладу схеми для першого PUB:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Розуміння необроблених даних часового розкладу
Хоча візуалізація даних часового розкладу схеми за допомогою методу `draw_circuit_schedule_timing` є найбільш поширеним варіантом використання, може бути корисним зрозуміти структуру повернутих необроблених даних часового розкладу. Це може допомогти, наприклад, програмно витягти інформацію.

Дані часового розкладу, що повертаються в `["compilation"]["scheduler_timing"]["timing"]`, є списком рядків. Кожен рядок представляє одну інструкцію на деякому каналі та розділяється комами на такі типи даних:

- `Branch` — Визначає, чи знаходиться інструкція у потоці керування (then / else) або в основній гілці.
- `Instruction` — Вентиль та кубіт для операції.
- `Channel` — Канал, якому призначається інструкція. Це може бути одним із таких:
      - `Qubit x` — Канал приводу для кубіта _x_.
      - `AWGRx_y` (довільний генератор хвильових форм для зчитування) — Використовується каналами зчитування для комунікації при вимірюванні кубітів. Аргументи _x_ та _y_ відповідають ідентифікатору інструмента зчитування та номеру кубіта відповідно.
- `T0` — Час початку інструкції в рамках повного розкладу
- `Duration` — Тривалість інструкції в одиницях _dt_ секунд, де 1 dt = 1 цикл планування. Значення `dt` для backend можна знайти за допомогою [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` — Тип імпульсної операції, що використовується.

Приклад:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Візуалізація часових розкладів
Починаючи з `qiskit-ibm-runtime` v0.43.0, можна візуалізувати часові розклади схем. Для візуалізації часових розкладів спочатку потрібно конвертувати метадані результату в `fig` за допомогою [методу `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26). Цей метод повертає рисунок `plotly`, який можна відображати безпосередньо, зберігати у файл або обидва варіанти. Докладніше про команди `plotly` для використання дивись у [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) та [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Наведення курсора на вихід показує таку інформацію, як початок, кінець та тривалість.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Приклад згенерованого рисунку')
#### Розуміння згенерованого рисунку
Зображення даних часового розкладу схеми, виведене `draw_circuit_schedule_timing`, передає таку інформацію:

- Вісь X — це час в одиницях _dt_ секунд, де 1 dt = 1 цикл планування. Значення `dt` для backend можна знайти за допомогою [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- Вісь Y — це канал (канали можна уявити як інструменти, що генерують імпульси).
    - `Receive channel` — Єдиний канал, що не є окремим інструментом. Це інструкція, виконувана на всіх каналах, що є частиною процедури зв'язку з хабом у цей час.
    - `Qubit x` — Канал приводу для кубіта x.
    - `AWGRx_y` (довільний генератор хвильових форм для зчитування) — Використовується каналами зчитування для комунікації при вимірюванні кубітів. Аргументи _x_ та _y_ відповідають ідентифікатору інструмента зчитування та номеру кубіта відповідно.
    - `Hub` — Керує мовленням.

Крім того, кожна інструкція має формат *X_Y*, де *X* — назва інструкції, а *Y* — тип імпульсу. Тип `play` застосовує керуючі імпульси, а `capture` записує стан кубіта. Можна також навести курсор на кожну інструкцію для отримання додаткових деталей. Наприклад, на попередньому рисунку показано керуючий імпульс для вентиля X, застосованого до кубіта 10 на 1161 dt.
### Наскрізний приклад
Цей приклад показує, як увімкнути опцію, отримати інформацію про часовий розклад схеми з метаданих та відобразити її як зображення.

Спочатку налаштуй середовище, визнач схеми та конвертуй їх у ISA-схеми, а потім визнач та запусти завдання.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Далі отримай часовий розклад схеми: